# Exploring DuckDB & SQLMesh

Exploring FHIR data infrastructure with DuckDB and SQLMesh

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 02/03/2026 (SGT)   | Martin | Create   | Notebook created. DuckDB exploration | 
| 17/03/2026 (SGT)   | Martin | Update   | Exploration of post-pipeline tables | 

# Content

* [DuckDB](#duckdb)

# DuckDB

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import os
from dotenv import load_dotenv

load_dotenv()

False

## Merging files

For systems that cannot handle docker files, use this to create a single merged DB

In [2]:
base_path = "../data/warehouse"
source_files = os.listdir(base_path)
source_files = [f"{base_path}/{file}" for file in source_files]
print(source_files)
target_db_path = "../data/merged_database.duckdb"

con = duckdb.connect(target_db_path)
con.execute("CREATE SCHEMA IF NOT EXISTS bronze;")

for i, file_path in enumerate(source_files):
  con.execute(f"ATTACH '{file_path}' AS source{i}")

  tables = con.execute(f"SELECT table_name FROM information_schema.tables WHERE table_catalog = 'source{i}'").fetchall()
  print(tables)
  for table_row in tables:
    table_name = table_row[0]
    table_name = f"bronze.{table_name}"

    con.execute(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM source{i}.{table_name} LIMIT 0")
    con.execute(f"INSERT INTO {table_name} SELECT * FROM source{i}.{table_name}")

for i, file_path in enumerate(source_files):
  con.execute(f"DETACH source{i}")

con.close()

print(f"Data from {source_files} has been merged into {target_db_path}")

['../data/warehouse/mimic_fhir.duckdb', '../data/warehouse/mimic_fhir_condition.duckdb', '../data/warehouse/mimic_fhir_disp_org_obs.duckdb', '../data/warehouse/mimic_fhir_encountered.duckdb', '../data/warehouse/mimic_fhir_location.duckdb', '../data/warehouse/mimic_fhir_obs.duckdb', '../data/warehouse/mimic_fhir_patient.duckdb']
[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('fhir_resources',)]
[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('fhir_resources',)]
[('fhir_resources',)]
[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[('fhir_resources',)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data from ['../data/warehouse/mimic_fhir.duckdb', '../data/warehouse/mimic_fhir_condition.duckdb', '../data/warehouse/mimic_fhir_disp_org_obs.duckdb', '../data/warehouse/mimic_fhir_encountered.duckdb', '../data/warehouse/mimic_fhir_location.duckdb', '../data/warehouse/mimic_fhir_obs.duckdb', '../data/warehouse/mimic_fhir_patient.duckdb'] has been merged into ../data/merged_database.duckdb


In [ ]:
con = duckdb.connect(source_files[1])
tables = con.sql("SHOW TABLES FROM bronze").df()
print(tables)
con.close()

# Exploring Tables

Bronze tables:

- `bronze.fhir_resources`: Collection of raw FHIR resource data
- `bronze.synthea`: Synthea generated data

Silver tables:

- `silver.obs_vital`: Patient vitals recorded in long format

Intermediate tables:

- `intermediate.vitals_wide`: Flattened vitals table containing patient recorded vitals (e.g heart rate, temperature, ...)

Gold (marts) tables:

- `marts.vitals_eda`: 

In [11]:
con = duckdb.connect("../data/warehouse/mimic_fhir.duckdb")
dbs = con.sql("SELECT * FROM duckdb_databases").df()
dbs

,database_name,database_oid,path,comment,tags,internal,type,readonly,encrypted,cipher
0,mimic_fhir,685,/Users/martz/Repos/MIMIC-FHIR/notebooks/../dat...,None,{'storage_version': 'v1.0.0+'},False,duckdb,False,False,None


In [12]:
tables = con.sql("SHOW ALL TABLES").df()
tables

,database,schema,name,column_names,column_types,temporary
0,mimic_fhir,bronze,fhir_resources,"[resource_type, resource_id, resource, source_...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR]",False
1,mimic_fhir,bronze,synthea,"[full_url, resource_type, resource, request, s...","[VARCHAR, VARCHAR, JSON, JSON, VARCHAR, VARCHAR]",False
2,mimic_fhir,intermediate,vitals_wide,"[encounter_id, patient_id, effective_datetime,...","[VARCHAR, VARCHAR, TIMESTAMP, DOUBLE, DOUBLE, ...",False
3,mimic_fhir,marts,vitals_eda,"[encounter_id, patient_id, effective_datetime,...","[VARCHAR, VARCHAR, TIMESTAMP, DOUBLE, DOUBLE, ...",False
4,mimic_fhir,silver,obs_vitals,"[resource_id, patient_id, encounter_id, effect...","[VARCHAR, VARCHAR, VARCHAR, TIMESTAMP, VARCHAR...",False
5,mimic_fhir,sqlmesh,_auto_restatements,"[snapshot_name, snapshot_version, next_auto_re...","[VARCHAR, VARCHAR, BIGINT]",False
6,mimic_fhir,sqlmesh,_environment_statements,"[environment_name, plan_id, environment_statem...","[VARCHAR, VARCHAR, VARCHAR]",False
7,mimic_fhir,sqlmesh,_environments,"[name, snapshots, start_at, end_at, plan_id, p...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
8,mimic_fhir,sqlmesh,_intervals,"[id, created_ts, name, identifier, version, st...","[VARCHAR, BIGINT, VARCHAR, VARCHAR, VARCHAR, B...",False
9,mimic_fhir,sqlmesh,_snapshots,"[name, identifier, version, snapshot, kind_nam...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False


In [8]:
files = con.sql("SELECT DISTINCT source_file FROM bronze.fhir_resources")
files

┌────────────────────────────────┐
│          source_file           │
│            varchar             │
├────────────────────────────────┤
│ Organization.ndjson            │
│ ObservationED.ndjson           │
│ Location.ndjson                │
│ MedicationDispenseED.ndjson    │
│ ConditionED.ndjson             │
│ ProcedureED.ndjson             │
│ ObservationVitalSignsED.ndjson │
│ EncounterED.ndjson             │
│ Patient.ndjson                 │
└────────────────────────────────┘

In [9]:
con.sql("SELECT * FROM bronze.synthea LIMIT 10")

┌───────────────────────────────────────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## Silver & Gold tables

Exploring Silver and Gold (mart) tables

In [20]:
# silver.obs_vital
con.sql("""
  SELECT *
  FROM silver.obs_vitals
  LIMIT 10
""")

┌──────────────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬────────┬────────────────┐
│             resource_id              │              patient_id              │             encounter_id             │ effective_datetime  │ loinc_code │ value  │      unit      │
│               varchar                │               varchar                │               varchar                │      timestamp      │  varchar   │ double │    varchar     │
├──────────────────────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼────────────┼────────┼────────────────┤
│ e6944724-62ed-53df-b56e-f26d77f8fc14 │ 000000ba-735e-5858-a92d-b856d73dd69a │ ab67e413-be5f-5c55-97d2-615c1941d950 │ 2169-09-06 17:35:00 │ 9279-1     │   18.0 │ breaths/minute │
│ 0df7928c-9982-5ca9-ab43-b965e5ec275f │ 000000ba-735e-5858-a92d-b856d73dd69a │ ab67e413-be5f-5c55-9

In [ ]:
# intermediate.vitals_wide
con.sql("SELECT * FROM intermediate.vitals_wide LIMIT 10")

┌──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬──────────┬──────────┬────────────┬───────────┬───────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────┬─────────────┐
│             encounter_id             │              patient_id              │ effective_datetime  │ temp_value │ hr_value │ rr_value │ spo2_value │ sbp_value │ dbp_value │ temp_present │ hr_present │ rr_present │ spo2_present │ sbp_present │ dbp_present │
│               varchar                │               varchar                │      timestamp      │   double   │  double  │  double  │   double   │  double   │  double   │    int32     │   int32    │   int32    │    int32     │    int32    │    int32    │
├──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼────────────┼──────────┼──────────┼────────────┼───────────┼───────────┼──────────────┼────────────┼────────────┼────────────

In [17]:
con.sql("""
  SELECT * 
  FROM intermediate.vitals_wide
  WHERE patient_id='fb348f58-e2e7-5fab-8118-494943e3b84b'
  ORDER BY effective_datetime DESC
  LIMIT 10
""")


┌──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬──────────┬──────────┬────────────┬───────────┬───────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────┬─────────────┐
│             encounter_id             │              patient_id              │ effective_datetime  │ temp_value │ hr_value │ rr_value │ spo2_value │ sbp_value │ dbp_value │ temp_present │ hr_present │ rr_present │ spo2_present │ sbp_present │ dbp_present │
│               varchar                │               varchar                │      timestamp      │   double   │  double  │  double  │   double   │  double   │  double   │    int32     │   int32    │   int32    │    int32     │    int32    │    int32    │
├──────────────────────────────────────┼──────────────────────────────────────┼─────────────────────┼────────────┼──────────┼──────────┼────────────┼───────────┼───────────┼──────────────┼────────────┼────────────┼────────────

In [21]:
# marts.vitals_eda
con.sql("""
  SELECT *
  FROM marts.vitals_eda
  LIMIT 10
""")

┌──────────────────────────────────────┬──────────────────────────────────────┬─────────────────────┬────────────┬──────────┬──────────┬────────────┬───────────┬───────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────┬─────────────┬────────────┬────────────┬────────────┬─────────────┬─────────────┬──────────────┬────────────────────────┬───────────┬────────────────────────┬─────────────────┐
│             encounter_id             │              patient_id              │ effective_datetime  │ temp_value │ hr_value │ rr_value │ spo2_value │ sbp_value │ dbp_value │ temp_present │ hr_present │ rr_present │ spo2_present │ sbp_present │ dbp_present │ n_vitals_6 │ n_vitals_5 │ n_vitals_3 │ hour_of_day │ day_of_week │ obs_position │ total_obs_in_encounter │ delta_min │ encounter_duration_hrs │ encounter_phase │
│               varchar                │               varchar                │      timestamp      │   double   │  double  │  double  │   double   │  double   

In [10]:
con.close()

In [ ]:
%load_ext watermark
%watermark